# Spatial Copilot — YOLO11-seg fine-tune on Colab

Runtime → **Change runtime type** → GPU (T4 is enough) before running.

Same steps as `COLAB.md`. The dataset is rebuilt from HuggingFace; nothing large is in git.

## 1. Clone the repo + install training deps

Do **not** `pip install -r requirements.txt` here — it pins CPU torch and would replace Colab's CUDA build.

In [ ]:
!git clone https://github.com/joshyajith863/Spatial_model.git
%cd Spatial_model
!pip install -q ultralytics pandas pyarrow huggingface_hub pyyaml
!nvidia-smi

## 2. Build the CubiCasa5K YOLO dataset

~1.5 GB download, 5–10 min. Writes `data/yolo/data.yaml` with the local root.

In [ ]:
!python scripts/convert_cubicasa_yolo.py --split both

## 3. Smoke test (200 images, 2 epochs)

Confirms the dataset/model/GPU path before burning an hour.

In [ ]:
!python scripts/train_yolo.py --device 0 --epochs 2 --batch 16 --imgsz 640 --limit 200

## 4. Persist weights to Drive, then train for real

`runs/` is gitignored and Colab's VM is temporary — write into Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Loader variant: live per-epoch progress bars.
!python main.py train --device 0 --epochs 50 --batch 16 \
    --project /content/drive/MyDrive/spatial_runs

## 5. Evaluate per class (floor / wall / door / window)

In [ ]:
!python scripts/eval_yolo.py --weights /content/drive/MyDrive/spatial_runs/cubicasa/weights/best.pt

## 6. Download the weights to your laptop

Put `best.pt` in the repo's `weights/` folder, then locally:
`python main.py load yolo --weights weights/best.pt`

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/spatial_runs/cubicasa/weights/best.pt')